## GPQA-D

#### Experiments:
- Testees : gpt-4.1-mini, qwen2.5-7b-it
- Judges : qwen3-4b
1) Baseline - free text
2) Baseline-wrong - all deliberate wrong answers
3) Multiple Answers
    - A. Forward
    - B. Backward
4) Surface Manipulation

In [1]:
from metrics_gpqa import mean_accuracy, calc_asr, decision_flip, significance, cohens
from tabulate import tabulate
import pandas as pd
import textwrap
import plotly.express as px
import os

def wrap_text(s, width=11):
    if isinstance(s, str):
        return "\n".join(textwrap.wrap(s, width=width))
    return s
def get_exp(path):
    judge = path.split("\\")[1]
    path = path.split("\\")[-1]
    if "gpt" in path.lower():
        testee = "gpt-4.1-mini"
    if "qwen" in path.lower():
        testee = "qwen2.5-7B-it"
    
    if "qual" in path.lower():
        qtype = "Qualitative"
    if "quant" in path.lower():
        qtype = "Quantitative"

    if "wrong" in path.lower():
        exp = "baseline-incorrect"
        
    elif "forward" in path.lower():
        exp = "multiple-forward"
    elif "backward" in path.lower():
        exp = "multiple-backward"
    elif "strategic" in path.lower():
        exp = "strategic"
    elif "verbose" in path.lower():
        exp = "verbose"
    else:
        exp = "baseline"
    return testee, judge, exp, qtype



In [2]:
def get_basic_metrics(scores):

    results = []
    
    for path in scores:
        # print(path)
        testee, judge, exp, qtype = get_exp(path)

        df = pd.read_csv(path)
        acc, num_samples = mean_accuracy(df)

        base_asr_qual, total_base_qual = calc_asr(path)

        # print(num_samples == total_base_qual)
        results.append([judge, testee, exp, qtype, num_samples, acc])
    headers = ["Judge", "Testee", "Experiment", "Q-Type", "Data Samples", "Accuracy"]
    table = tabulate(results, headers=headers, tablefmt="grid")
    print(table)
    return results


In [3]:
#baseline, attack pair paths to scores

def get_comparisons(comparisons):
    
    results = []
    # judge = "qwen3-4B"
    for base, game in comparisons:
        # print(base, game)
        base1 = base.split("/")[-1]
        if "qual" in base1.lower():
            qtype = "Qualitative"
        if "quant" in base1.lower():
            qtype = "Quantitative"
        if "gpt" in base1.lower():
            testee = "gpt-4.1-mini"
        if "qwen" in base1.lower():
            testee = "qwen2.5-7B-it"
        exp = ""
        for path in [base, game]:
            judge = path.split("/")[2]
            if "wrong" in path.lower():
                exp += "baseline-incorrect/"
            elif "forward" in path.lower():
                exp += "multiple-forward/"
            elif "backward" in path.lower():
                exp += "multiple-backward/"
            elif "strategic" in path.lower():
                exp += "strategic/"
            elif "verbose" in path.lower():
                exp += "verbose/"
            else:
                exp += "baseline/"

        # bdf = pd.read_csv(base)
        # acc, num_samples = mean_accuracy(bdf)

        base_asr, total_base = calc_asr(base)
        game_asr, total_game = calc_asr(game)
        base_suc = base_asr*total_base
        game_successes = game_asr*total_game

        flips_asr, flip_successes, game_samples = decision_flip(base, game)
        
        #base successes, base_total_num, game_successes, game_total_num
        base_asr, attack_asr, zstat, pval = significance(base_suc, total_base, game_successes, game_samples)
        
        coh = cohens(base, game)
        cohensd = coh['cohen_d']
        cohensh = coh['cohen_h']

        percent = (game_asr - base_asr) / base_asr

        results.append([judge, testee, exp, qtype, flip_successes, flips_asr, pval, cohensd, base_asr, attack_asr, total_base, game_samples, base_suc, game_successes,  percent ])
        

    wrapped = [[wrap_text(cell) for cell in row] for row in results]
    headers = ["Judge", "Testee", "Experiment", "Q-Type","Decision Flips", "Decision Flip %", "p-value", "Cohen's d","Base ASR", "Gamed ASR", "Base Samples", "Gamed Samples", "Base Successes", "Gamed Successes", "Percent change"]
    table = tabulate(results, headers=headers, tablefmt="grid")
    print(table)
    return results



In [4]:
exprs = ["baseline", "strategic", "forward", 
         "verbose"
        ]
btypes = ["qual", "quant"]
testees = ["gpt", "qwen"]
judges = ["gpt4.1mini", "qwen2.5_7b",
          "qwen3_4b", "gemma2b"
          ]
# bench = "gpqa_cont"
# for expr in exprs:
#     for testee in testees:
#         for judge in judges:
#             for btype in btypes:        
#                 scores.append(f"scores/{bench}/{judge}/{expr}/{testee}_{btype}_{expr}_scores.csv")
scores = {}
paradigms = ["gpqa_bin", "gpqa_bin_judge", "gpqa_cont", "gpqa_cont_judge"]
for para in paradigms:
    all_files = []
    for root, _, filenames in os.walk("scores/"+para):
        for filename in filenames:
            if "summary" not in filename and "backward" not in filename and "surface" not in filename and "wrong" not in filename:
                all_files.append(os.path.join(root, filename))

    scores[para] = [a for a in all_files if a.endswith("csv")]



# judges = ["gpt4.1mini", "qwen2.5_7b",
#         #   "qwen3_4b", "gemma2b"
#           ]
# bench = "gpqa_bin_judge"
# scores_j = []
# for expr in exprs:
#     for testee in testees:
#         for judge in judges:
#             for btype in btypes:        
#                 scores_j.append(f"scores/{bench}/{judge}/{expr}/{testee}_{btype}_{expr}_scores.csv")



In [24]:
results = {}
for para in paradigms:
    res = get_basic_metrics(scores[para])
    results[para] = res



+------------+---------------+------------------+--------------+----------------+------------+
| Judge      | Testee        | Experiment       | Q-Type       |   Data Samples |   Accuracy |
+============+===============+==================+==============+================+============+
| gemma2b    | gpt-4.1-mini  | baseline         | Qualitative  |            106 |  0.971698  |
+------------+---------------+------------------+--------------+----------------+------------+
| gemma2b    | gpt-4.1-mini  | baseline         | Quantitative |             92 |  1         |
+------------+---------------+------------------+--------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | baseline         | Qualitative  |            106 |  0.915094  |
+------------+---------------+------------------+--------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | baseline         | Quantitative |             92 |  0.913043  |
+------------+---------------+------------------+-

In [ ]:
# exprs = os.listdir("scores/gpqa_cont_judge/qwen2.5_7b")
# paths = []
# for e in exprs:
#     temp = (os.listdir("scores/gpqa_cont_judge/qwen2.5_7b/"+e))
#     temp = ["scores/gpqa_cont_judge/qwen2.5_7b/"+e+"/"+i for i in temp]
#     paths.extend(temp)
# for p in paths[1:]:
#     df = pd.read_csv(p)
#     # df['score_raw'] = df["score"]
#     df["score"] = df["score_raw"].str.extract(r'score\"*[: ]+([0-9.]+)').astype(float)
#     # print(df.head())
#     df.to_csv(p)



In [41]:
#mean_Accuracies
def get_avg_acc_over_bench(qual_scores, title):
    means = []
    print(qual_scores[0])
    
    for score in qual_scores:
        df1 = pd.read_csv(score)
        df2 = pd.read_csv(score.replace("qual", "quant"))
        mean_acc, tot = mean_accuracy(pd.concat([df1, df2], ignore_index=True))
        testee, judge, exp, qtype = get_exp(score)
        means.append([judge, testee, exp, tot, mean_acc])

    headers = ["Judge", "Testee", "Experiment", "Data Samples", "Accuracy"]
    table = tabulate(means, headers=headers, tablefmt="grid")
    print(table)

    df = pd.DataFrame(means, columns=["Judge", "Testee","Experiment", "Data Samples", "Accuracy"])
    fig = px.bar(
    df,
    x="Experiment",
    y="Accuracy",
    color="Testee",
    barmode="group",
    facet_col="Judge",
    text="Accuracy",
    title=title,
    color_discrete_map={
        "gpt-4.1-mini": "#8c564b",
        "qwen2.5-7B-it": "#70ad47"
        }
    )

    # Clean up facet labels
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

    # Adjust subplot spacing
    fig.update_layout(
        uniformtext_minsize=8,
        uniformtext_mode='hide',
        width=750,    # wider figure for readability
        height=500,
        margin=dict(t=80, b=80),
    )

    # Optional: rotate x-axis labels
    fig.update_xaxes(tickangle=30)

    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig.show()

    return df

titles = [
    "Binary Matcher", "Binary Judge", "Continuous Matcher", "Continuous Judge"
]
mean_dfs = {}
for i, para in enumerate(paradigms):
    qual_scores = [s for s in scores[para] if "qual" in s]
    mean_m_df = get_avg_acc_over_bench(qual_scores, 
                                       f"Average Accuracies - GPQA-D - {titles[i]}")

    mean_dfs[para] = mean_m_df

scores/gpqa_bin\gemma2b\baseline\gpt_qual_baseline_scores.csv
+------------+---------------+------------------+----------------+------------+
| Judge      | Testee        | Experiment       |   Data Samples |   Accuracy |
+============+===============+==================+================+============+
| gemma2b    | gpt-4.1-mini  | baseline         |            198 |  0.984848  |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | baseline         |            198 |  0.914141  |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | gpt-4.1-mini  | multiple-forward |            198 |  0.888889  |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | multiple-forward |            198 |  0.914141  |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | gpt-4.1-mini  | strategic        |         

scores/gpqa_bin_judge\gpt4.1mini\baseline\gpt_qual_baseline_scores.csv
+------------+---------------+------------------+----------------+------------+
| Judge      | Testee        | Experiment       |   Data Samples |   Accuracy |
+============+===============+==================+================+============+
| gpt4.1mini | gpt-4.1-mini  | baseline         |            198 |   0.626263 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | qwen2.5-7B-it | baseline         |            198 |   0.227273 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | gpt-4.1-mini  | multiple-forward |            198 |   0.141414 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | qwen2.5-7B-it | multiple-forward |            198 |   0.111111 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | gpt-4.1-mini  | strategic        |

scores/gpqa_cont\gemma2b\baseline\gpt_qual_baseline_scores.csv
+------------+---------------+------------------+----------------+------------+
| Judge      | Testee        | Experiment       |   Data Samples |   Accuracy |
+============+===============+==================+================+============+
| gemma2b    | gpt-4.1-mini  | baseline         |            198 |   0.675859 |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | baseline         |            198 |   0.557475 |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | gpt-4.1-mini  | multiple-forward |            198 |   0.393182 |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | qwen2.5-7B-it | multiple-forward |            198 |   0.406717 |
+------------+---------------+------------------+----------------+------------+
| gemma2b    | gpt-4.1-mini  | strategic        |        

scores/gpqa_cont_judge\gpt4.1mini\baseline\gpt_qual_baseline_scores.csv
+------------+---------------+------------------+----------------+------------+
| Judge      | Testee        | Experiment       |   Data Samples |   Accuracy |
+============+===============+==================+================+============+
| gpt4.1mini | gpt-4.1-mini  | baseline         |            198 |   0.808247 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | qwen2.5-7B-it | baseline         |            198 |   0.448438 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | gpt-4.1-mini  | multiple-forward |            198 |   0.437563 |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | qwen2.5-7B-it | multiple-forward |            198 |   0.30203  |
+------------+---------------+------------------+----------------+------------+
| gpt4.1mini | gpt-4.1-mini  | strategic        

In [72]:
# res_j = get_basic_metrics(scores_j)

In [27]:
def plot_judge_matcher(df, title):
    fig = px.bar(
        df,
        x="Experiment",
        y="Accuracy",
        color="Paradigm",
        barmode="group",
        facet_col="Testee",  # optional: create separate plots per Testee
        title=title
    )

    fig.update_layout(
        xaxis_title="Experiment",
        yaxis_title="Accuracy",
        legend_title="Paradigm"
    )

    fig.show()

In [44]:
for i, para in enumerate(paradigms):
    res = results[para]
    title = titles[i]
    df = pd.DataFrame(res, columns=["Judge", "Testee","Experiment", "Q-Type", "DataPoints", "Accuracy"])
    df["Model_Qtype"] = df["Testee"] + " (" + df["Q-Type"] + ")"

    fig = px.bar(
        df,
        width=750,
        height=500,
        x="Experiment",
        y="Accuracy",
        color="Model_Qtype",   # distinct color for each Model + Q-Type
        barmode="group",
        facet_col="Judge",
        text="Accuracy",
        title=f"Accuracy per Experiment - {title}"
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
    fig.show()



In [37]:
para

'gpqa_cont_judge'

In [43]:
for i in range(0, 4, 2):
    res1 = mean_dfs[paradigms[i]]
    res2 = mean_dfs[paradigms[i+1]]
    title1 = titles[i]
    title2 = titles[i+1]
    columns = ["Judge", "Testee", "Experiment", "Data Samples", "Count", "Accuracy"]

    m_df = pd.DataFrame(res1, columns=columns)
    m_df['Paradigm'] = 'Matcher'
    j_df = pd.DataFrame(res2, columns=columns)
    j_df['Paradigm'] = 'Judge'

    combined_df = pd.concat([j_df, m_df], ignore_index=True)

    gpt_df = combined_df[combined_df["Judge"]=="gpt4.1mini"]
    qwen_df = combined_df[combined_df["Judge"]=="qwen2.5_7b"]

    plot_judge_matcher(gpt_df, f"Accuracy - {title2} vs {title1} - Judge Model = GPT 4.1 Mini")
    plot_judge_matcher(qwen_df, f"Accuracy - {title2} vs {title1} - Judge Model = Qwen2.5-7B-It")


In [73]:
comparisons = []
pairs = [("baseline", "strategic"), ("baseline", "forward"), ("forward", "strategic"), 
        #  ("baseline", "verbose")
        ]
btypes = ["qual", "quant"]
testees = ["gpt", "qwen"]
bench = "gpqa_bin"
judges = ["gpt4.1mini", "qwen2.5_7b",
          "qwen3_4b", "gemma2b"
          ]
for p1, p2 in pairs:
    for judge in judges:
        for testee in testees:
            for btype in btypes:
                comparisons.append((f"scores/{bench}/{judge}/{p1}/{testee}_{btype}_{p1}_scores.csv",
                                    f"scores/{bench}/{judge}/{p2}/{testee}_{btype}_{p2}_scores.csv"))
            
results_c = get_comparisons(comparisons)

+------------+---------------+-----------------------------+--------------+------------------+-------------------+-------------+-------------+------------+-------------+----------------+-----------------+------------------+-------------------+------------------+
| Judge      | Testee        | Experiment                  | Q-Type       |   Decision Flips |   Decision Flip % |     p-value |   Cohen's d |   Base ASR |   Gamed ASR |   Base Samples |   Gamed Samples |   Base Successes |   Gamed Successes |   Percent change |
+============+===============+=============================+==============+==================+===================+=============+=============+============+=============+================+=================+==================+===================+==================+
| gpt4.1mini | gpt-4.1-mini  | baseline/strategic/         | Qualitative  |                4 |         0.0377358 | 0.343523    |  -0.129771  |  0.179245  |   0.132075  |            106 |             106 |       

In [74]:
comparisons = []
pairs = [("baseline", "strategic"), ("baseline", "forward"), ("forward", "strategic"), ("baseline", "verbose")]
btypes = ["qual", "quant"]
testees = ["gpt", "qwen"]
bench = "gpqa_bin_judge"
judges = ["gpt4.1mini", "qwen2.5_7b"]
for p1, p2 in pairs:
    for judge in judges:
        for testee in testees:
            for btype in btypes:
                comparisons.append((f"scores/{bench}/{judge}/{p1}/{testee}_{btype}_{p1}_scores.csv",
                                    f"scores/{bench}/{judge}/{p2}/{testee}_{btype}_{p2}_scores.csv"))
            
results_c = get_comparisons(comparisons)

"""
Note: Might want to report normalised scores - ie, metrics like decision flips and p-val/ essentially with baseline since that would account for exactly how strict the matcher is to begin with (to differentiate between how much is just strict scoring and how much is a drop in susceptibility for baseline-gaming with scale scores)
"""


+------------+---------------+-----------------------------+--------------+------------------+-------------------+-------------+-------------+------------+-------------+----------------+-----------------+------------------+-------------------+------------------+
| Judge      | Testee        | Experiment                  | Q-Type       |   Decision Flips |   Decision Flip % |     p-value |   Cohen's d |   Base ASR |   Gamed ASR |   Base Samples |   Gamed Samples |   Base Successes |   Gamed Successes |   Percent change |
+============+===============+=============================+==============+==================+===================+=============+=============+============+=============+================+=================+==================+===================+==================+
| gpt4.1mini | gpt-4.1-mini  | baseline/strategic/         | Qualitative  |                6 |        0.0566038  | 0.000154494 |  -0.535682  |  0.726415  |   0.471698  |            106 |             106 |       

'\nNote: Might want to report normalised scores - ie, metrics like decision flips and p-val/ essentially with baseline since that would account for exactly how strict the matcher is to begin with (to differentiate between how much is just strict scoring and how much is a drop in susceptibility for baseline-gaming with scale scores)\n'

In [138]:

def plot_judge_matcher(df, title):
    fig = px.bar(
        df,
        x="Experiment",
        y="Accuracy",
        color="Paradigm",
        barmode="group",
        facet_col="Testee",  # optional: create separate plots per Testee
        title=title
    )

    fig.update_layout(
        xaxis_title="Experiment",
        yaxis_title="Accuracy",
        legend_title="Paradigm"
    )

    fig.show()

